<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Main_Notebook_Biochar_Forest_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:

# ============================================================
# MAIN NOTEBOOK — Biochar Forest Estimation Pipeline
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [40]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

In [41]:
# ── CELL 2: Mount Google Drive + define paths ─────────────
from google.colab import drive
drive.mount('/content/drive')

# Drive paths — for GEE raw exports
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
GEE_FOLDER   = DRIVE_FOLDER + 'GEE_exports/'
FAO_FOLDER   = DRIVE_FOLDER + 'FAO_data/'

# Repo paths — for final cleaned results
REPO_FOLDER  = '/content/Biochar_forest_estimation/'
DATA_FOLDER  = REPO_FOLDER + 'data/'

print('✅ Google Drive mounted')
print('✅ Paths defined')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Paths defined


In [42]:
# ── CELL 3: Authenticate GEE ──────────────────────────────────────────────────
import ee
import os
os.environ['GOOGLE_MAPS_API_KEY'] = ''

ee.Authenticate()   # ← uncomment ONLY if token expired
ee.Initialize(project='spry-blade-487218-n0')

print('✅ GEE connected')

✅ GEE connected


In [ ]:
# ── CELL 4: Clone GitHub Repo ─────────────────────────────
%cd /content/          # ← always start from here

import getpass, subprocess
PAT = getpass.getpass('Enter your GitHub PAT: ')

result = subprocess.run(
    f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    %cd /content/Biochar_forest_estimation/
    print('✅ Repo cloned')
else:
    print('❌ Clone failed')
    print(result.stderr)

[Errno 2] No such file or directory: '/content/ # ← always start from here'
/content/Biochar_forest_estimation/Biochar_forest_estimation


In [ ]:
 #── CELL 5: Run Notebook 1 — GEE Computation ─────────────────────────────────
%run Notebook_1_GEE_Forest_Area_Computation.ipynb

print('✅ Notebook 1 complete — tasks submitted to GEE')

In [ ]:
# ── CELL 6: Monitor GEE Tasks ─────────────────────────────────────────────────
# ⚠️ Run this cell manually — repeat until all tasks show COMPLETED
import time

for i in range(30):                          # check up to 30 times
    print(f'\n── Check {i+1} ──')
    all_done = True

    print('Country tasks:')
    for task in country_tasks:
        status = task.status()
        print(f"  {status['description']}: {status['state']}")
        if status['state'] not in ['COMPLETED', 'FAILED']:
            all_done = False

    print('State tasks:')
    for task in state_tasks:
        status = task.status()
        print(f"  {status['description']}: {status['state']}")
        if status['state'] not in ['COMPLETED', 'FAILED']:
            all_done = False

    if all_done:
        print('\n✅ All tasks completed!')
        break

    time.sleep(60)

attention to the name of the notebook 2
it should be fixed

In [ ]:
# ── CELL 7: export GEE exports created from Notebook 1 and stored on Google drive to github────────────────────────────────────────
import shutil

# copy raw GEE exports from Drive to repo data folder
files = [
    'forest_area_10.csv',    'forest_area_20.csv', 'forest_area_30.csv','forest_area_40.csv',  'forest_area_50.csv',
    'states_forest_area_10.csv', 'states_forest_area_20.csv','states_forest_area_30.csv', 'states_forest_area_40.csv',  'states_forest_area_50.csv'
        ]

for f in files:
    shutil.copy(GEE_FOLDER + f, DATA_FOLDER + f)
    print(f'✅ Copied {f} to repo')

In [ ]:
import shutil

# copy raw GEE exports from Drive to repo data folder
files = [
    'forest_area_10.csv', 'forest_area_20.csv', 'forest_area_30.csv', 'forest_area_40.csv', 'forest_area_50.csv',
    'states_forest_area_10.csv', 'states_forest_area_20.csv','states_forest_area_30.csv', 'states_forest_area_40.csv','states_forest_area_50.csv'
        ]

for f in files:
    shutil.copy(GEE_FOLDER + f, DATA_FOLDER + f)
    print(f'✅ Copied {f} to repo')

In [ ]:
# ── CELL 8: Run Notebook 2 — Analysis ────────────────────────────────────────
%run Notebook_2_Forest_Area_Analysis.ipynb
#test if push and commit work

print('✅ Notebook 2 complete — results ready')

In [ ]:
# ── CELL 9: Push Results to GitHub ───────────────────────────────────────────
import subprocess

# pull first to get latest changes
!git pull https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main

# then push
!git add .
!git commit -m "update forest area results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Results pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)